# NOAA GFS — Download

**Model:** NOAA Global Forecast System  
**Type:** Physics-based deterministic  
**Cycles:** 00Z, 06Z, 12Z, 18Z (all full 384h)  
**Library:** `herbie-data`

## Files produced

```
data/YYYY-MM-DD_HHZ/
    msl_f024.grib2           T+24h verification snapshots
    sfc_f024.grib2
    pl_f024.grib2
    wave_f024.grib2
    sfc_f000-f120_3h.grib2   Group 1 — surface (3-hourly, 0–120h)
    pl_f000-f120_12h.grib2   Group 2 — upper air (12-hourly, 0–120h)
    wave_f000-f120_3h.grib2  Group 3 — wave (gfs_wave global.0p25, 3-hourly)
```

## GFS → ECMWF parameter name mapping

| GFS GRIB | cfgrib key | ECMWF equivalent |
|----------|------------|------------------|
| PRMSL | `prmsl` | `msl` |
| PWAT | `pwat` | `tcwv` |
| WATR | `watr` | `ro` |
| GUST | `gust` | `fg10` |
| CAPE surface | `cape` | real CAPE (not MUCAPE) |
| ABSV | `absv` | `vo` (absolute vs relative) |
| TCDC | `tcc` | `tcc` (already in %, not 0–1) |

In [ ]:
from herbie import Herbie
from pathlib import Path
import pandas as pd

BASE_DIR = Path("data")
BASE_DIR.mkdir(parents=True, exist_ok=True)
print("Ready. BASE_DIR:", BASE_DIR.resolve())

In [ ]:
def find_latest_gfs(max_lookback_h: int = 36) -> pd.Timestamp:
    now = pd.Timestamp.now("UTC").replace(tzinfo=None)
    for lag_h in range(0, max_lookback_h + 1):
        candidate = (now - pd.Timedelta(hours=lag_h)).floor("6h")
        try:
            H = Herbie(candidate, model="gfs", product="pgrb2.0p25", fxx=0, verbose=False)
            inv = H.inventory()
            if inv is not None and len(inv) > 0:
                return candidate
        except Exception:
            continue
    raise RuntimeError(f"No GFS run found in the last {max_lookback_h} h")

latest_run  = find_latest_gfs()
latest_time = latest_run.hour

run_label  = f"{latest_run.strftime('%Y-%m-%d')}_{latest_time:02d}Z"
output_dir = BASE_DIR / run_label
output_dir.mkdir(parents=True, exist_ok=True)

print("Latest run  :", latest_run)
print("Cycle (UTC) :", latest_time)
print("Output dir  :", output_dir.resolve())

In [ ]:
# ── Download helper ───────────────────────────────────────────────────────────
tmp_dir = output_dir / ".tmp"
tmp_dir.mkdir(exist_ok=True)

def download_step(fxx, search, target, model="gfs", product="pgrb2.0p25"):
    H = Herbie(latest_run, model=model, product=product, fxx=fxx, verbose=False)
    tmp = Path(H.download(search, save_dir=tmp_dir))
    target.unlink(missing_ok=True)
    tmp.rename(target)
    return target

def download_batch(steps, search, target, model="gfs", product="pgrb2.0p25"):
    parts = []
    for fxx in steps:
        H = Herbie(latest_run, model=model, product=product, fxx=fxx, verbose=False)
        p = Path(H.download(search, save_dir=tmp_dir))
        parts.append(p)
        print(f"  f{fxx:03d} ✓", end="  ")
    print()
    with open(target, "wb") as out:
        for p in parts: out.write(p.read_bytes())
    for p in parts: p.unlink(missing_ok=True)
    print(f"Saved: {target.name}  ({target.stat().st_size // 1024} KB)")

# Verification snapshots
download_step(24, ":PRMSL:mean sea level:", output_dir / "msl_f024.grib2")
download_step(24,
    ":TMP:2 m above ground:|:PRMSL:mean sea level:|:UGRD:10 m above ground:|:VGRD:10 m above ground:",
    output_dir / "sfc_f024.grib2")
download_step(24, ":(HGT|TMP|UGRD|VGRD):(850|700|500|300|250) mb:", output_dir / "pl_f024.grib2")
download_step(24, ":(HTSGW|WDIR|PERPW):", output_dir / "wave_f024.grib2",
              model="gfs_wave", product="global.0p25")
print("Snapshots saved.")

In [ ]:
# ── Group 1: Surface — 3-hourly, 0–120h ──────────────────────────────────────
steps_3h = list(range(0, 121, 3))

SFC = {}
SFC["thermo"]      = (":TMP:2 m above ground:|"
                      ":DPT:2 m above ground:|"
                      ":TMP:surface:|"
                      ":SPFH:2 m above ground:")
SFC["wind"]        = (":UGRD:10 m above ground:|"
                      ":VGRD:10 m above ground:|"
                      ":GUST:surface:|"
                      ":VWSH:surface:")
SFC["pressure"]    = (":PRMSL:mean sea level:|"
                      ":PRES:surface:")
SFC["moisture"]    = (":APCP:surface:|"
                      ":ACPCP:surface:|"
                      ":WATR:surface:|"
                      ":PWAT:entire atmosphere (considered as a single layer):")
SFC["cloud"]       = (":TCDC:entire atmosphere (considered as a single layer):|"
                      ":LCDC:low cloud layer:|"
                      ":MCDC:mid cloud layer:|"
                      ":HCDC:high cloud layer:")
SFC["energy"]      = (":LHTFL:surface:|"
                      ":SHTFL:surface:|"
                      ":ULWRF:top of atmosphere:|"
                      ":DSWRF:surface:|"
                      ":DLWRF:surface:")
SFC["boundary"]    = ":HPBL:surface:"
SFC["instability"] = (":CAPE:surface:|"
                      ":CIN:surface:|"
                      ":LFTX:500-1000 mb:|"
                      ":4LFTX:lowest 500-1000 mb:")

SFC_SEARCH = "|".join(SFC.values())
print(f"Downloading {len(steps_3h)} steps — surface  ({len(SFC)} groups)")
download_batch(steps_3h, SFC_SEARCH, output_dir / "sfc_f000-f120_3h.grib2")
print("Group 1 saved.")

In [ ]:
# ── Group 1b: Composite reflectivity — 3-hourly (cycle-dependent) ─────────────
# REFC is not always present in pgrb2.0p25; wrap in try/except so it doesn't
# abort the download session if the field is absent this cycle.
try:
    download_batch(steps_3h, ":REFC:entire atmosphere:",
                   output_dir / "refc_f000-f120_3h.grib2")
    print("Group 1b saved: refc_f000-f120_3h.grib2")
except Exception as e:
    print(f"Group 1b (REFC): not available this cycle — {e}")

In [ ]:
# ── Group 2: Upper-air — 12-hourly, 0–120h ───────────────────────────────────
PL_SEARCH = (
    ":(HGT|TMP|UGRD|VGRD|VVEL|SPFH|RH|ABSV):"
    "(1000|925|850|700|600|500|400|300|250|200) mb:"
)
print(f"Downloading {len(list(range(0, 121, 12)))} steps — upper air")
download_batch(list(range(0, 121, 12)), PL_SEARCH, output_dir / "pl_f000-f120_12h.grib2")
print("Group 2 saved.")

In [ ]:
# ── Group 3: Wave — 3-hourly, 0–120h (gfs_wave global.0p25) ──────────────────
print(f"Downloading {len(steps_3h)} steps — wave")
download_batch(steps_3h, ":(HTSGW|WDIR|PERPW):",
               output_dir / "wave_f000-f120_3h.grib2",
               model="gfs_wave", product="global.0p25")
print("Group 3 saved.")